# NB01 - Designing Datasets for Dashboards

## Mise en place
Vérifie si le lab a été mis en place.
En cas d'erreur lancer la cellule de mise en place dans le notebook **NB00 - Setup et Exploration**

In [0]:
%run "./Resources/NB01/Setup"

## A. Création et nommage du Dashboard

Un **dashboard** dans Databricks permet de combiner plusieurs visualisations (graphiques, tables, widgets) issues de requêtes SQL dans une vue unique et interactive.

### Créer un dashboard

1. Dans le menu latéral de votre **workspace**, cliquez sur **Dashboards** (ou **SQL Warehouses → Dashboards** selon la version).
2. Cliquez sur le bouton **+ Create Dashboard**.
3. Un dashboard vide s'ouvre dans l'éditeur.

### Ajouter des éléments au dashboard

- Cliquez sur **+ Add** pour ajouter :
  - Une **visualization** (graphique basé sur une requête SQL),
  - Un **text** (titre, description, note),
  - Un **filter** (filtre global applicable à plusieurs visualisations).
- Chaque visualisation est liée à une **requête SQL** que vous écrivez ou réutilisez depuis le SQL Editor.

### Nommer le dashboard

1. Cliquez sur le nom par défaut (souvent *Untitled dashboard*) en haut de la page.
2. Saisissez un nom **descriptif et explicite**, par exemple :
   - `Sales Performance — Q3 2024`
   - `Monitoring Pipeline Latency`
3. Appuyez sur **Entrée** ou cliquez en dehors du champ pour valider.

### Bonnes pratiques de nommage

| Bonne pratique | Exemple |
|---|---|
| Inclure le domaine ou le cas d'usage | `Marketing Campaigns ROI` |
| Ajouter la période si pertinent | `Inventory Snapshot 2024` |
| Utiliser un préfixe de groupe d'équipe | `DATAENG — Quality Gates` |
| Éviter les noms génériques | ❌ `Dashboard 1` → ✅ `Customer Churn Analysis` |

> 💡 Le nom du dashboard est visible dans la liste des dashboards du workspace ; un nom clair facilite la recherche et le partage avec vos collaborateurs.


## B. Ajouter des données
- Cliquer sur l'onglet "Data" en dessous du nom du Dashboard
- il y a 2 options :
    - add data
    - add SQL dataset (on utilise celui là)

### B1. Dataset pour la table *ca_opportunities*
- Dans l'onglet data, cliquer sur **Add SQL dataset**
- Choisir le catalog et schema : 
    - catalog : labuser_<username>
    - Schema : aibi_dashboards
- inserer la requête suivante : `SELECT * FROM ca_opportunities`
- Cliquer sur **Run** pour exécuter la requête et examiner les resultats
- Renomer le dataset en **Opportunités**

#### B1.1 Ajout d'un calcul spécifique
Disons que l'on veut catégoriser les opportunités en 3 catégories :
- Small
- Medium
- Large
On peut le faire de deux manière :
1. Modifier directement la requête SQL du dataset lors de l'import : 
```SQL
SELECT
    *,
    CASE NTILE(3) OVER (ORDER BY estimated_value)
        WHEN 1 THEN 'Small'
        WHEN 2 THEN 'Medium'
        WHEN 3 THEN 'Large'
    END AS op_size
FROM ca_opportunities;
```
2. Au dessus du resultat lors du premier run, cliquer sur "Add custom calculation"
    - Nommer le calcul : op_size
    - et rentrer le code SQL suivant : 
    ```SQL
    CASE NTILE(3) OVER (ORDER BY estimated_value)
        WHEN 1 THEN 'Small'
        WHEN 2 THEN 'Medium'
        WHEN 3 THEN 'Large'
    END
    ```


### B2. Données de commandes
- Cliquer sur l'onglet Data
- "Add SQL dataset"
- Choisir le catalog et schema : 
    - catalog : labuser_<username>
    - Schema : aibi_dashboards
- insérer la requête suivante : 
```SQL
select
    c.company_name,
    c.region,
    o.order_date,
    YEAR(o.order_date) as year,
    p.product_name,
    o.amount,
    o.quantity,
    s.salesrep_name
from ca_orders as o
    inner join ca_customers as c
    on o.customer_id = c.customer_id
    inner join ca_products as p
    on o.product_id = p.product_id
    inner join ca_sales_reps as s
    on o.salesrep_id = s.salesrep_id;
```


## C. Créations des Visualisations

### C1. En tête
Dans un premier temps on va créer l'en-tête du Dashboard : 
1. Cliquer sur "add a textbox" et le positioner en haut du dashboard (ce bloc utilise le markdown)
```
# Sales France
Vue d'ensemble des performances commerciales à travers le suivi des ventes, des commandes et des clients. Ce dashboard permet d'analyser les principaux KPI, leur évolution dans le temps et l'atteinte des objectifs de vente.

```

### C2. KPI - Somme total des ventes
1. Cliquer sur "add a Visualization" et la positioner en dessous de l'en-tête
2. Dans le nouveau menue sur la droite, cocher "Title" et "Description"
3. Dans "Visualization" choisire "Counter"
4. Pour la valeur : 
    - Dans "value" choisir : objectifs -> actual_sales
    - Transform -> None
    - Style : ajouter 2 "conditional style" :
        1. `>=` laisser la police en vert
        2. `<` mettre la police en rouge
    - Format :
        1. Choisir "custom"
        2. Type -> $ et ensuite choisir EUR
    - Row -> 1 (année en cours)
5. Pour la comparaison :
    - Dans "value" choisir : objectifs -> target
        - Transform -> None
        - Format :
            1. Choisir "custom"
            2. Type -> $ et ensuite choisir EUR
        - Row -> 1 (année en cours)
6. Donner un titre et une description au visuel

### C3. KPI - Total des commandes
1. Cliquer sur "add a Visualization" et la positioner en dessous de l'en-tête
2. Dans le nouveau menue sur la droite, cocher "Title" et "Description"
3. Dans "Visualization" choisire "Counter"
4. Pour la valeur : 
    - Dans "value" choisir : commandes -> COUNT(*)

### C3. KPI - Total des commandes
1. Cliquer sur "add a Visualization" et la positioner en dessous de l'en-tête
2. Dans le nouveau menue sur la droite, cocher "Title" et "Description"
3. Dans "Visualization" choisire "Counter"
4. Pour la valeur : 
    - Dans "value" choisir : commandes -> quantity
    - Transform -> SUM

### C4. Graphique combo
1. Cliquer sur "add a Visualization" et la positioner en dessous de l'en-tête
2. Dans le nouveau menue sur la droite, cocher "Title" et "Description"
3. Dans "Visualization" choisire "Combo"
4. Pour la X-Axis : 
    - Dans "X-Axis" choisir : objectifs -> target_year
5. Pour la Y-Axis :
    - Dans "Bar" choisir : objectifs -> actual_sales
        - Transform -> SUM
    - Dans "Line" choisir : objectifs -> target
        - Transform -> SUM
6. Donner un titre

### C5. Area Chart avec Genie
Pour la création de ce dernier visuel on va utiliser Genie code : 
1. Cliquer sur "add a Visualization" et la positioner a la suite
2. lorsque le visuel est placé il y a une zone de texte modifiable et remplir a l'intérieur : 
    - **Create an area chart of total sales by month and color by product name. Add clean display values. Format the y axis using euros**
    - ou en Français : **Créer un graphique en aires du total des ventes par mois, avec une couleur distincte par nom de produit. Afficher les valeurs de manière claire et formater l'axe Y en euros (€).**
3. au bout d'un certain Genie devrait avoir modifier le nouveau visuel avec la configuration suivante : 
    - X-Axis -> MONTHLY(order_date)
    - Y-Axis -> SUM(amount)
    - Color -> product_name

Et le dashboard devrait ressembler a ça :
    ![dashboard1.png](./Resources/dashboard1.png "dashboard1.png")